# Week 8 — Agents II: Memory + Multi-step + Stabilizers
### *Making agents more capable (memory) without making them unstable or unsafe.*

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/week8_agents_ii_memory.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Learning objectives
By the end of these two class sessions, you can:
- Explain the difference between **short-term state** and **long-term memory** in an agent.
- Build a tiny **memory store** using embeddings (store notes + retrieve relevant notes).
- Describe why loops become unstable and name 3 **stabilizers** (budgets, stop rules, progress checks).
- Sketch an agent loop: **Plan → Act → Observe → Store/Retrieve → Stop**.
- Understand where to add **verification steps** for higher reliability.


In [ ]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git || true
import sys, platform, re
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

sys.path.append("/content/main")
from course_utils import get_text_embedding

# Optional: DSPy
try:
    import dspy
except Exception:
    dspy = None

print(f"✅ Ready! Python {platform.python_version()} | dspy={'yes' if dspy else 'no'}")


---

# Tue 8 — Memory as an engineering component

## **What did Lab 7 teach us?**
We saw systems can be tricked through:
- user text
- retrieved docs
- tool outputs

Now we’ll upgrade our agents — but carefully.

### Discussion
> If an agent can store memory, what new risks appear?

Write 2 risks:
1)  
2)  

---

## **What is “memory” in an agent?**
Two common types:

### 1) Short-term state (scratchpad)
- notes in the current conversation or run
- reset every run
- helps keep steps coherent

### 2) Long-term memory (store)
- persists across tasks
- can be retrieved later (often using embeddings)
- can be wrong, stale, or unsafe

We’ll build a tiny memory store (notes + embedding search).


In [ ]:
# @title Build a tiny memory store (notes + embedding retrieval)
def normalize(v):
    v = np.array(v, dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

class TinyMemory:
    def __init__(self):
        self.notes = []   # list of dicts: {"text":..., "tag":...}
        self.X = None     # embedding matrix

    def add(self, text, tag=""):
        self.notes.append({"text": text, "tag": tag})
        emb = normalize(get_text_embedding(text))
        self.X = emb[None, :] if self.X is None else np.vstack([self.X, emb])

    def search(self, query, k=3):
        if self.X is None:
            return []
        q = normalize(get_text_embedding(query))
        sims = self.X @ q
        idx = np.argsort(-sims)[:k]
        return [self.notes[int(i)] for i in idx]

mem = TinyMemory()
mem.add("Student prefers short answers with bullet points.", tag="preference")
mem.add("Project name is 'Bluebird' and deadline is Feb 1.", tag="project")
mem.add("Interns may join on-call only after manager approval.", tag="policy")

print(mem.search("When is Bluebird due?", k=2))


### Reflection
> Why is embedding-based memory retrieval useful?  
> What could go wrong if the memory note is wrong?

---

## **Memory turns agents into control loops with state**
A minimal loop can be:

1) Plan next step  
2) Act (call tool)  
3) Observe result  
4) Update state  
5) Decide stop vs continue

With memory:
- you may **store** a note
- you may **retrieve** old notes

Let’s visualize the loop with memory included.


In [ ]:
# @title Unifying Diagram v7: Agent loop with memory store
G = nx.DiGraph()
for n in ["User", "Planner", "Tools", "Observation", "Short-term State", "Memory Store", "Final Answer"]:
    G.add_node(n)

edges = [
    ("User","Planner"),
    ("Planner","Tools"),
    ("Tools","Observation"),
    ("Observation","Short-term State"),
    ("Short-term State","Planner"),
    ("Planner","Memory Store"),
    ("Memory Store","Planner"),
    ("Planner","Final Answer"),
]
G.add_edges_from(edges)

plt.figure(figsize=(9,4))
pos = nx.spring_layout(G, seed=8)
nx.draw(G, pos, with_labels=True, node_size=2200, font_size=9, arrows=True)
plt.title("Unifying Diagram v7: Agent loop + memory store")
plt.tight_layout()
plt.show()


---

# Thu 8 — Stabilizers + verification

## **Why do loops become unstable?**
Two reasons show up constantly:
1) **No stop rule** (“keep going forever”)
2) **No progress check** (“repeat the same step”)

### Common failure patterns
- retry spiral
- ping-pong between tools
- amplified wrong assumption (“memory of a mistake”)

We’ll introduce stabilizers that are easy to implement.


## **Stabilizers (simple, practical)**
1) **Step budget**: max steps per run  
2) **Tool budget**: max tool calls per run  
3) **Stop conditions**: clear definition of “done”  
4) **Progress checks**: “did we learn something new this step?”  
5) **Deterministic debug mode**: avoid randomness while testing

We can model “budgeting” as a safety rail: it doesn’t make you correct, but it prevents runaway failure.


In [ ]:
# @title Toy demo: step budget vs no budget (loop that might not converge)
def toy_agent(noise=0.3, max_steps=None):
    # Goal: reach state near 0 by subtracting an estimate.
    x = 10.0
    steps = 0
    history = [x]
    while abs(x) > 0.5:
        steps += 1
        # "estimate" is noisy -> can overshoot
        estimate = x + np.random.normal(0, noise)
        x = x - 0.9 * estimate
        history.append(x)
        if max_steps is not None and steps >= max_steps:
            break
    return history

plt.figure(figsize=(7,3))
h1 = toy_agent(max_steps=None)
h2 = toy_agent(max_steps=8)
plt.plot(h1, label="no budget")
plt.plot(h2, label="budgeted (8 steps)")
plt.axhline(0, linewidth=1)
plt.title("Budgets cap unstable behavior")
plt.xlabel("step")
plt.ylabel("state")
plt.legend()
plt.tight_layout()
plt.show()


### Reflection
> Budgets can stop runaway loops, but they can also stop you before success.  
> How would you choose a budget?

---

## **Verification patterns**
When stakes are high, add a verification step:
- **doc claims**: quote the supporting line + cite chunk id
- **math**: recompute with calculator
- **tool actions**: ask for confirmation before side effects

This is a big theme: **agents are systems**. We can add checkpoints.

---

## (Optional) DSPy: multi-step as a structured module
DSPy is helpful when you want:
- a clear input/output contract for each step
- reproducibility and evaluation

We’ll only show a tiny sketch.


In [ ]:
# @title Optional: DSPy sketch of a memory-aware step
if dspy is None:
    print("DSPy not installed (ok).")
else:
    class NextAction(dspy.Signature):
        goal: str = dspy.InputField()
        memory: str = dspy.InputField(desc="Retrieved notes (untrusted).")
        action: str = dspy.OutputField(desc="One of: CALL_RAG, CALL_CALC, STORE_NOTE, ANSWER, STOP")
        note: str = dspy.OutputField(desc="If STORE_NOTE, the note text; else empty.")

    print("Defined DSPy Signature:", NextAction.__name__)


---

## Lab 8 preview: build a memory-backed agent
In Lab 8 you’ll:
- store notes
- retrieve notes
- add stabilizers (step/tool budgets)
- evaluate when memory helps vs hurts

---

<details>
<summary><strong>Instructor Notes</strong></summary>

### Tue pacing
- 0–12: Lab 7 review (attack categories + causes)
- 12–25: memory types + TinyMemory demo
- 25–40: loop diagram v7 + discussion of risks
- 40–50: preview Lab 8 tasks

### Thu pacing
- 0–10: instability failure modes
- 10–25: stabilizers list + toy budget demo
- 25–40: verification patterns
- 40–50: Lab 8 kickoff

</details>
